In [ ]:
!pip install sentence-transformers pandas numpy scikit-learn -q

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv('job_descriptions.csv',
                 on_bad_lines='skip',
                 quoting=3,
                 engine='python',
                 nrows=25000)

print("Колонки:", df.columns.tolist())
df.head(2)



Колонки: ['Job Id', 'Experience', 'Qualifications', 'Salary Range', 'location', 'Country', 'latitude', 'longitude', 'Work Type', 'Company Size', 'Job Posting Date', 'Preference', 'Contact Person', 'Contact', 'Job Title', 'Role', 'Job Portal', 'Job Description', 'Benefits', 'skills', 'Responsibilities', 'Company', 'Company Profile']


,,,,,,,,,,,,,,,,,,Job Id,Experience,Qualifications,Salary Range,location,Country,latitude,longitude,Work Type,Company Size,...,Contact,Job Title,Role,Job Portal,Job Description,Benefits,skills,Responsibilities,Company,Company Profile
1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,2022-04-24,Female,Brandon Cunningham,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,"""Social Media Managers oversee an organizations social media presence. They create and schedule content",engage with followers,and analyze social media metrics to drive bra...,"""{'Flexible Spending Accounts (FSAs)",Relocation Assistance,Legal Assistance,Employee Recognition Programs,"Financial Counseling'}""","""Social media platforms (e.g.",Facebook,Twitter,...,and interact with the online community. Devel...,Icahn Enterprises,"""{""""Sector"""":""""Diversified""""","""""Industry"""":""""Diversified Financials""""","""""City"""":""""Sunny Isles Beach""""","""""State"""":""""Florida""""","""""Zip"""":""""33160""""","""""Website"""":""""www.ielp.com""""","""""Ticker"""":""""IEP""""","""""CEO"""":""""David Willetts""""}"""
398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,2022-12-19,Female,Francisco Larsen,461-509-4216,Web Developer,Frontend Web Developer,Idealist,"""Frontend Web Developers design and implement user interfaces for websites",ensuring they are visually appealing and user...,"""{'Health Insurance",Retirement Plans,Paid Time Off (PTO),Flexible Work Arrangements,"Employee Assistance Programs (EAP)'}""","""HTML",CSS,JavaScript Frontend frameworks (e.g.,React,...,PNC Financial Services Group,"""{""""Sector"""":""""Financial Services""""","""""Industry"""":""""Commercial Banks""""","""""City"""":""""Pittsburgh""""","""""State"""":""""Pennsylvania""""","""""Zip"""":""""15222""""","""""Website"""":""""www.pnc.com""""","""""Ticker"""":""""PNC""""","""""CEO"""":""""William S. Demchak""""}""",None


In [ ]:
text_cols = ['Job Title', 'Job Description', 'skills', 'Qualifications', 'Responsibilities']

df['search_text'] = (
    df['Job Title'].fillna('Job') + ' ' +
    df['skills'].fillna('').str[:200] + ' ' +
    df['Qualifications'].fillna('').str[:150]
).str.strip()
print(f"Создано {len(df):,} текстов")
print("Пример:")
df[['Job Title', 'search_text']].head()


Создано 22,384 текстов
Пример:


,,,,,,,,,,,,,,,,,,Job Title,search_text
1089843540111562,5 to 15 Years,M.Tech,$59K-$99K,Douglas,Isle of Man,54.2361,-4.5481,Intern,26801,2022-04-24,Female,Brandon Cunningham,001-381-930-7517x737,Digital Marketing Specialist,Social Media Manager,Snagajob,"""Social Media Managers oversee an organizations social media presence. They create and schedule content",Icahn Enterprises,"Icahn Enterprises """"Zip"""":""""33160"""" ""{'Flexibl..."
398454096642776,2 to 12 Years,BCA,$56K-$116K,Ashgabat,Turkmenistan,38.9697,59.5563,Intern,100340,2022-12-19,Female,Francisco Larsen,461-509-4216,Web Developer,Frontend Web Developer,Idealist,"""Frontend Web Developers design and implement user interfaces for websites","""{""""Sector"""":""""Financial Services""""","""{""""Sector"""":""""Financial Services"""" """"Website""..."
481640072963533,0 to 12 Years,PhD,$61K-$104K,Macao,"""Macao SAR","China""",22.1987,113.5439,Temporary,84525,2022-09-14,Male,Gary Gibson,9687619505,Operations Manager,Quality Control Manager,Jobs2Careers,"""""City"""":""""San Antonio""""","""""City"""":""""San Antonio"""" """"CEO"""":""""Wayne Peaco..."
116831420231957,4 to 12 Years,MCA,$59K-$93K,Brussels,Belgium,50.5039,4.4699,Full-Time,23196,2023-07-25,Male,Matthew Gill,(973)791-5355x52199,Software Tester,Quality Assurance Analyst,Snagajob,"""A Quality Assurance Analyst tests software and products to ensure they meet quality standards. They identify defects","""{""""Sector"""":""""Infrastructure""""","""{""""Sector"""":""""Infrastructure"""" """"Website"""":""""..."
1292168246729889,3 to 15 Years,PhD,$63K-$103K,George Town,Cayman Islands,19.3133,-81.2546,Temporary,26119,2023-04-10,Both,Zachary Hansen,001-268-510-4362x789,Teacher,Classroom Teacher,FlexJobs,"""A Classroom Teacher educates students in a specific subject or grade level. They create lesson plans","Package and Freight Delivery""""","Package and Freight Delivery"""" """"Ticker"""":""""FD..."


In [ ]:
!pip install sentence-transformers scikit-learn -q

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import time


In [ ]:
models = {
    'MiniLM-L6': 'sentence-transformers/all-MiniLM-L6-v2',
    'bge-m3': 'BAAI/bge-m3',
    'MultiLM': 'sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2'
}

sample = df['search_text'].head(10).tolist()
results = []

for name, path in models.items():
    print(f"Тест {name}...")
    t0 = time.time()
    model = SentenceTransformer(path)
    emb = model.encode(sample)
    sim = cosine_similarity([emb[0]], emb)[0][0]
    results.append([name, emb.shape[1], f"{time.time()-t0:.1f}s", f"{sim:.3f}"])

pd.DataFrame(results, columns=['модель', 'dim', 'время', 'similarity']).style.background_gradient()


Тест MiniLM-L6...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Тест bge-m3...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Тест MultiLM...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


,модель,dim,время,similarity
0,MiniLM-L6,384,2.1s,1.000
1,bge-m3,1024,6.4s,1.000
2,MultiLM,384,4.4s,1.000


In [ ]:

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

emb = model.encode(df['search_text'].tolist(), show_progress_bar=True)
print(emb.shape)



Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/700 [00:00<?, ?it/s]

(22384, 384)


In [ ]:
def search_v2(query, top_k=5):
    query = f"job: {query} skills:"

    q_emb = model.encode([query])
    scores = cosine_similarity(emb, q_emb).flatten()
    scores = (scores - scores.min()) / (scores.max() - scores.min())

    top_idx = np.argsort(scores)[-top_k:][::-1]
    results = df.iloc[top_idx][['Job Title', 'skills']].copy()
    results['score'] = scores[top_idx]
    return results

search_v2("Python developer")


,,,,,,,,,,,,,,,,,,Job Title,skills,score
2594337392461923,1 to 10 Years,MBA,$60K-$99K,Jakarta,Indonesia,-0.7893,113.9213,Part-Time,129532,2023-04-21,Female,Julie Torres,(626)986-1942x9230,Network Administrator,Systems Administrator,USAJOBS,"""Manage and maintain an organizations IT infrastructure",NaN,None,1.000000
1977756979575342,3 to 15 Years,MBA,$62K-$102K,"""Malabo (de jure)","""",Equatorial Guinea,1.6508,10.2679,Contract,37452,2022-04-29,Female,Rose Ochoa,672-662-0401,Data Entry Clerk,Record Keeper,LinkedIn,NaN,None,0.935421
1224062168170043,1 to 11 Years,B.Tech,$58K-$85K,Freetown,Sierra Leone,8.4606,-11.7799,Full-Time,121421,2022-06-25,Male,Donald Marsh,765.497.1559x0426,Structural Engineer,Bridge Engineer,Monster,"""A Bridge Engineer specializes in designing and maintaining bridges",None,None,0.814941
2622433632230243,5 to 11 Years,M.Tech,$65K-$108K,Riyadh,Saudi Arabia,23.8859,45.0792,Full-Time,61994,2022-03-15,Female,Wendy Johnson,001-636-878-9902x413,Customer Success Manager,Account Manager,Idealist,"""An Account Manager builds and maintains relationships with clients or customers","""""Zip"""":""""6902""""",None,0.793060
139405868973356,2 to 15 Years,B.Tech,$65K-$101K,Ottawa,Canada,56.1304,-106.3468,Temporary,46437,2022-01-04,Male,Christopher Farley,7572836837,Physical Therapist,Geriatric Physical Therapist,The Muse,"""Focus on the physical therapy needs of elderly patients","""""Zip"""":""""N/A""""",None,0.791479


In [ ]:
queries_v2 = ["Python Developer", "Data Scientist", "Software Engineer"]
sims_v2 = [search_v2(q)['score'].mean() for q in queries_v2]
print(f"cosine similarity: {np.mean(sims_v2):.3f}")


cosine similarity: 0.835
